# HAM10000: ResNet50 vs ViT-B/16

This project compares ResNet50 and ViT-B/16 for 7-class skin-lesion classification on HAM10000.

This notebook is the revision of `skincancer.ipynb`. Task 1 records the baseline. Task 2 builds the lesion-level split. Task 3 builds the image pipeline. Task 4 adds class balancing. Task 5 trains a fresh ResNet50. The test set stays unused until both fair checkpoints exist.

## Official baseline (image-level split)

These figures are taken from the `classification_report` outputs in `skincancer.ipynb`. They replace the handwritten summary tables in that notebook.

The split behind this table is a random split of image rows (7,010 train / 1,502 validation / 1,503 test). The same `lesion_id` can appear in more than one split, so the scores are optimistic.

**This table is not comparable to the new lesion-level split.** It is the "before" record only. It is not the target to beat, and it is not the final claim.

| Run | Model | Test accuracy | Macro F1 | Weighted F1 | Melanoma recall |
|---|---|---:|---:|---:|---:|
| 1. Cross-entropy, 10 epochs, fresh ImageNet weights | ResNet50 | 82% | 0.67 | 0.81 | 0.50 |
| 1. Cross-entropy, 10 epochs, fresh ImageNet weights | ViT-B/16 | 76% | 0.53 | 0.73 | 0.43 |
| 2. Focal loss, continued from run 1 weights | ResNet50 | 84% | 0.73 | 0.84 | 0.53 |
| 2. Focal loss, continued from run 1 weights | ViT-B/16 | 76% | 0.54 | 0.75 | 0.42 |
| 3. Focal loss + MixUp, fresh ResNet50 only | ResNet50 | 84% | 0.71 | 0.83 | 0.51 |

Run 2 did not start from ImageNet again. It kept training the run 1 weights. Run 3 trained a new ResNet50 only; there is no MixUp ViT in the baseline.

## Metric policy

Every experiment prints all of the following, computed in code:

- Accuracy
- Balanced accuracy
- Macro F1
- Weighted F1
- Per-class precision, recall, and F1
- Melanoma recall
- Melanoma average precision
- Top 10 confusion pairs

Class names, in label order, stay fixed: `nv`, `mel`, `bkl`, `bcc`, `akiec`, `vasc`, `df`.

Accuracy is reported, and it is not the headline. Nevus is about 67% of HAM10000, so a model that always predicts nevus already scores about 67% accuracy. Macro F1 and melanoma recall carry the comparison.

## Checkpoint rule

Save the epoch with the highest **validation macro F1**.

Validation loss is plotted. It is not used to choose the checkpoint. Early stopping, when it is added, also watches validation macro F1.

## Test-set rule

No test evaluation until both fair checkpoints exist: a fresh ResNet50 and a fresh ViT-B/16, each selected by validation macro F1 on the lesion-level split.

Until that point, reported scores are validation scores only. The test set is then evaluated once for those two checkpoints.

## Task 2 — Lesion-level split

Images are assigned by `lesion_id`, not by image row. Each lesion has one diagnosis. Lesions are stratified with `random_state=42`: 70% train, then the remaining 30% split evenly into validation and test. Every image of a lesion stays in that lesion's split.

The fixed label map is `nv=0`, `mel=1`, `bkl=2`, `bcc=3`, `akiec=4`, `vasc=5`, `df=6`.

The cell saves `split_train.csv`, `split_val.csv`, and `split_test.csv` only after it maps all 10,015 images to a real jpg in `HAM10000_images_part_1` or `HAM10000_images_part_2` and finds zero missing paths. The saved columns are `lesion_id`, `image_id`, `dx`, and `path`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

LABEL_MAP = {
    "nv": 0,
    "mel": 1,
    "bkl": 2,
    "bcc": 3,
    "akiec": 4,
    "vasc": 5,
    "df": 6,
}

here = Path.cwd()
metadata_candidates = [
    here / "HAM10000_metadata.csv",
    Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"),
]
metadata_path = next((p for p in metadata_candidates if p.exists()), None)
if metadata_path is None:
    sys.exit("HAM10000_metadata.csv was not found. Split files were not saved.")

df = pd.read_csv(metadata_path)
print("Metadata rows:", len(df))
print("Unique lesions:", df["lesion_id"].nunique())
dx_per_lesion = df.groupby("lesion_id")["dx"].nunique()
print("Max diagnoses per lesion:", int(dx_per_lesion.max()))
if (dx_per_lesion > 1).any():
    sys.exit("A lesion has more than one dx. Split files were not saved.")

candidate_roots = [
    here,
    Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"),
]
folder_pairs = (
    ("HAM10000_images_part_1", "HAM10000_images_part_2"),
    ("ham10000_images_part_1", "ham10000_images_part_2"),
)

found = None
for root in candidate_roots:
    for part1_name, part2_name in folder_pairs:
        part1 = root / part1_name
        part2 = root / part2_name
        if part1.is_dir() and part2.is_dir():
            found = (part1, part2)
            break
    if found:
        break

if found is None:
    sys.exit(
        "Image folders were not found next to this notebook or under the Kaggle "
        "HAM10000 input path. Expected HAM10000_images_part_1 and "
        "HAM10000_images_part_2 (or the lowercase names). "
        "The metadata CSV does not say which folder holds each jpg, so paths "
        "cannot be confirmed. Split files were not saved."
    )

folder1, folder2 = found
print("Image folder 1:", folder1)
print("Image folder 2:", folder2)

image_paths = {}
for folder in (folder1, folder2):
    for img in folder.iterdir():
        if img.suffix.lower() == ".jpg":
            image_paths[img.stem] = str(img.resolve())

df["path"] = df["image_id"].map(image_paths)
missing = int(df["path"].isna().sum())
print("Paths mapped:", int(df["path"].notna().sum()))
print("Missing paths:", missing)
if len(df) != 10015 or missing != 0:
    print(df.loc[df["path"].isna(), "image_id"].head(20).to_string(index=False))
    sys.exit(
        f"Expected 10015 paths and 0 missing. Got rows={len(df)}, missing={missing}. "
        "Split files were not saved."
    )

lesions = df.groupby("lesion_id", as_index=False)["dx"].first()
try:
    train_lesions, temp_lesions = train_test_split(
        lesions, test_size=0.30, random_state=42, stratify=lesions["dx"]
    )
    val_lesions, test_lesions = train_test_split(
        temp_lesions, test_size=0.50, random_state=42, stratify=temp_lesions["dx"]
    )
except ValueError:
    print("Stratify failed. Lesion counts per class:")
    print(lesions["dx"].value_counts().to_string())
    sys.exit("Split files were not saved.")

train_ids = set(train_lesions["lesion_id"])
val_ids = set(val_lesions["lesion_id"])
test_ids = set(test_lesions["lesion_id"])

overlap_train_val = len(train_ids & val_ids)
overlap_train_test = len(train_ids & test_ids)
overlap_val_test = len(val_ids & test_ids)
print("Lesion overlap train-val:", overlap_train_val)
print("Lesion overlap train-test:", overlap_train_test)
print("Lesion overlap val-test:", overlap_val_test)

if overlap_train_val or overlap_train_test or overlap_val_test:
    sys.exit("Lesion overlap was not 0. Split files were not saved.")
if train_ids | val_ids | test_ids != set(df["lesion_id"]):
    sys.exit("The three splits do not cover every lesion. Split files were not saved.")

unknown = set(df["dx"]) - set(LABEL_MAP)
if unknown:
    sys.exit(f"Unknown dx values {unknown}. Split files were not saved.")

columns = ["lesion_id", "image_id", "dx", "path"]
split_frames = {
    "split_train.csv": df[df["lesion_id"].isin(train_ids)],
    "split_val.csv": df[df["lesion_id"].isin(val_ids)],
    "split_test.csv": df[df["lesion_id"].isin(test_ids)],
}

for name, part in split_frames.items():
    print()
    print(name)
    print("Images:", len(part))
    print("Lesions:", part["lesion_id"].nunique())
    print(part["dx"].value_counts().sort_index().to_string())
    part[columns].to_csv(here / name, index=False)

print()
print("LABEL_MAP", LABEL_MAP)
print("Saved split_train.csv, split_val.csv, split_test.csv")

Metadata rows: 10015
Unique lesions: 7470
Max diagnoses per lesion: 1
Image folder 1: D:\Projects\skincancer\HAM10000_images_part_1
Image folder 2: D:\Projects\skincancer\HAM10000_images_part_2
Paths mapped: 10015
Missing paths: 0
Lesion overlap train-val: 0
Lesion overlap train-test: 0
Lesion overlap val-test: 0

split_train.csv
Images: 6981
Lesions: 5229
dx
akiec     222
bcc       361
bkl       772
df         71
mel       773
nv       4683
vasc       99

split_val.csv
Images: 1532
Lesions: 1120
dx
akiec      53
bcc        82
bkl       160
df         24
mel       173
nv       1018
vasc       22

split_test.csv
Images: 1502
Lesions: 1121
dx
akiec      52
bcc        71
bkl       167
df         20
mel       167
nv       1004
vasc       21

LABEL_MAP {'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3, 'akiec': 4, 'vasc': 5, 'df': 6}
Saved split_train.csv, split_val.csv, split_test.csv


## Task 3 — Image pipeline

`SkinDataset` reads `split_train.csv`, `split_val.csv`, and `split_test.csv` from Task 2. Each sample opens the original JPG in the `path` column. Labels come from the fixed map `nv=0`, `mel=1`, `bkl=2`, `bcc=3`, `akiec=4`, `vasc=5`, `df=6`.

The training transform resizes to 224, then applies a horizontal flip, a rotation of up to 20 degrees, and color jitter, before converting to a tensor and normalizing with ImageNet mean `[0.485, 0.456, 0.406]` and std `[0.229, 0.224, 0.225]`. Validation and test use resize, tensor, and the same normalize only.

Loaders use batch size 32. Only the training loader is shuffled. This cell prints one training batch and does not train a model.

In [ ]:
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

LABEL_MAP = {
    "nv": 0,
    "mel": 1,
    "bkl": 2,
    "bcc": 3,
    "akiec": 4,
    "vasc": 5,
    "df": 6,
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class SkinDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform
        unknown = set(self.df["dx"]) - set(LABEL_MAP)
        if unknown:
            raise ValueError(f"Unknown dx values in {csv_path}: {unknown}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row["path"])
        if "preprocessed_images" in path.parts:
            raise ValueError(f"Refusing preprocessed image: {path}")
        image = Image.open(path).convert("RGB")
        label = LABEL_MAP[row["dx"]]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


here = Path.cwd()
train_dataset = SkinDataset(here / "split_train.csv", transform=train_transform)
val_dataset = SkinDataset(here / "split_val.csv", transform=eval_transform)
test_dataset = SkinDataset(here / "split_test.csv", transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print("Batch shape:", tuple(images.shape))
print("Label set:", sorted(set(labels.tolist())))
channel_mean = images.mean(dim=(0, 2, 3))
channel_std = images.std(dim=(0, 2, 3))
print("Per-channel mean:", [round(v, 4) for v in channel_mean.tolist()])
print("Per-channel std:", [round(v, 4) for v in channel_std.tolist()])


Batch shape: (32, 3, 224, 224)
Label set: [0, 2, 3]
Per-channel mean: [0.7356, 0.1837, 0.4698]
Per-channel std: [1.0169, 0.9424, 0.9904]


## Task 4 — Class balancing

Class weights are computed from `split_train.csv` only: `weight_c = N_train / (7 * count_c)`. The training loader draws images with a `WeightedRandomSampler` and `shuffle=False`. Validation and test loaders stay unweighted and unshuffled, as built in Task 3.

The loss for later training is weighted cross-entropy with label smoothing 0.1. This cell does not train and does not read the test split.

In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

CLASS_ORDER = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
train_counts = train_dataset.df["dx"].value_counts()
n_train = len(train_dataset)
class_weights = {
    name: n_train / (len(CLASS_ORDER) * int(train_counts[name]))
    for name in CLASS_ORDER
}

print(f"{'class':<8} {'count':>8} {'weight':>10}")
for name in CLASS_ORDER:
    print(f"{name:<8} {int(train_counts[name]):>8} {class_weights[name]:>10.4f}")

weight_tensor = torch.tensor(
    [class_weights[name] for name in CLASS_ORDER], dtype=torch.float32
)
sample_weights = train_dataset.df["dx"].map(class_weights).to_numpy()
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)
train_loader = DataLoader(
    train_dataset, batch_size=32, sampler=sampler, shuffle=False
)

sampled_idx = list(sampler)[:800]
sampled_dx = train_dataset.df.iloc[sampled_idx]["dx"]
print("\nSampled class histogram (800 draws):")
for name in CLASS_ORDER:
    print(f"{name:<8} {int((sampled_dx == name).sum()):>5}")

criterion = torch.nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=0.1)
print("\nCriterion:", criterion)
print("Loss weights:", [round(v, 4) for v in criterion.weight.tolist()])
print("Label smoothing:", criterion.label_smoothing)


class       count     weight
nv           4683     0.2130
mel           773     1.2901
bkl           772     1.2918
bcc           361     2.7626
akiec         222     4.4923
vasc           99    10.0736
df             71    14.0463

Sampled class histogram (800 draws):
nv          95
mel        105
bkl        102
bcc        119
akiec      131
vasc       119
df         129

Criterion: CrossEntropyLoss()
Loss weights: [0.213, 1.2901, 1.2918, 2.7626, 4.4923, 10.0736, 14.0463]
Label smoothing: 0.1


## Task 5 — Fresh ResNet50

ResNet50 starts from ImageNet weights. The final layer is `Linear(2048, 7)`. No checkpoint from the old notebook is loaded.

AdamW uses weight decay 0.01, learning rate `1e-4` on the backbone and `1e-3` on the new layer, with a cosine schedule for up to 25 epochs. The loss is the weighted cross-entropy with label smoothing 0.1 from Task 4.

The saved checkpoint is the epoch with the highest validation macro F1. Training stops after 5 epochs with no improvement. The test loader is not used.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torchvision.models import ResNet50_Weights, resnet50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model_resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model_resnet.fc = torch.nn.Linear(2048, 7)
model_resnet = model_resnet.to(device)
criterion = criterion.to(device)

backbone_params = [
    param for name, param in model_resnet.named_parameters() if not name.startswith("fc.")
]
optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-4},
        {"params": model_resnet.fc.parameters(), "lr": 1e-3},
    ],
    weight_decay=0.01,
)
epochs = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

train_losses, val_losses = [], []
val_accuracies, val_balanced, val_macro_f1s = [], [], []
best_macro_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
patience = 5
checkpoint_path = here / "best_resnet_fair.pth"

train_resized = [Image.open(path).convert("RGB").resize((224, 224)) for path in train_dataset.df["path"]]
val_resized = [Image.open(path).convert("RGB").resize((224, 224)) for path in val_dataset.df["path"]]
train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CachedSkin(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), int(self.labels[idx])

train_labels = train_dataset.df["dx"].map(LABEL_MAP).tolist()
val_labels = val_dataset.df["dx"].map(LABEL_MAP).tolist()
train_loader = DataLoader(
    CachedSkin(train_resized, train_labels, train_aug),
    batch_size=32,
    sampler=sampler,
    shuffle=False,
)
val_loader = DataLoader(
    CachedSkin(val_resized, val_labels, eval_tensor),
    batch_size=32,
    shuffle=False,
)
print(f"Cached images: train {len(train_resized)}, val {len(val_resized)}", flush=True)

for epoch in range(1, epochs + 1):
    model_resnet.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model_resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    model_resnet.eval()
    val_loss_sum = 0.0
    pred_list, label_list = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model_resnet(images)
            val_loss_sum += criterion(outputs, labels).item()
            pred_list.extend(outputs.argmax(dim=1).cpu().tolist())
            label_list.extend(labels.cpu().tolist())
    val_loss = val_loss_sum / len(val_loader)
    val_acc = accuracy_score(label_list, pred_list)
    val_bal = balanced_accuracy_score(label_list, pred_list)
    val_f1 = f1_score(label_list, pred_list, average="macro", zero_division=0)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_balanced.append(val_bal)
    val_macro_f1s.append(val_f1)
    scheduler.step()

    print(
        f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
        f"Val Bal Acc: {val_bal:.4f} | Val Macro F1: {val_f1:.4f}",
        flush=True,
    )

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model_resnet.state_dict(), checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch}", flush=True)
            break

epoch_axis = range(1, len(train_losses) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epoch_axis, train_losses, label="Train loss")
axes[0].plot(epoch_axis, val_losses, label="Val loss")
axes[0].set_title("ResNet50 — Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[1].plot(epoch_axis, val_macro_f1s, label="Val macro F1", color="green")
axes[1].set_title("ResNet50 — Validation macro F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()
fig.tight_layout()
fig.savefig(here / "resnet_fair_curves.png", dpi=150)
plt.show()

model_resnet.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
print(f"Best epoch: {best_epoch}")
print(f"Best val macro F1: {best_macro_f1:.4f}")


Device: cuda
Cached images: train 6981, val 1532
Epoch 1/25 | Train Loss: 0.6835 | Val Loss: 5.5574 | Val Acc: 0.4321 | Val Bal Acc: 0.6274 | Val Macro F1: 0.3762
Epoch 2/25 | Train Loss: 0.4913 | Val Loss: 5.5999 | Val Acc: 0.3662 | Val Bal Acc: 0.5997 | Val Macro F1: 0.3530
Epoch 3/25 | Train Loss: 0.4609 | Val Loss: 5.0780 | Val Acc: 0.2363 | Val Bal Acc: 0.6223 | Val Macro F1: 0.3360
Epoch 4/25 | Train Loss: 0.4394 | Val Loss: 4.8542 | Val Acc: 0.5679 | Val Bal Acc: 0.6626 | Val Macro F1: 0.4830
Epoch 5/25 | Train Loss: 0.4144 | Val Loss: 5.1248 | Val Acc: 0.4980 | Val Bal Acc: 0.6498 | Val Macro F1: 0.4406
Epoch 6/25 | Train Loss: 0.4153 | Val Loss: 5.3358 | Val Acc: 0.5555 | Val Bal Acc: 0.6677 | Val Macro F1: 0.5214
Epoch 7/25 | Train Loss: 0.4275 | Val Loss: 4.6171 | Val Acc: 0.6332 | Val Bal Acc: 0.6893 | Val Macro F1: 0.5557
Epoch 8/25 | Train Loss: 0.3870 | Val Loss: 4.5850 | Val Acc: 0.6482 | Val Bal Acc: 0.6657 | Val Macro F1: 0.5325
Epoch 9/25 | Train Loss: 0.4076 | Val L

## Task 5 rerun — sampler only

The training sampler from Task 4 stays. The loss is ordinary cross-entropy with label smoothing 0.1 and no class weights. This run does not load or overwrite `best_resnet_fair.pth`. The checkpoint is `best_resnet_sampler.pth`, chosen by validation macro F1. The test set is not used.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torchvision.models import ResNet50_Weights, resnet50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
print("Loss: unweighted cross-entropy, label smoothing", criterion.label_smoothing)

model_resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model_resnet.fc = torch.nn.Linear(2048, 7)
model_resnet = model_resnet.to(device)

backbone_params = [
    param for name, param in model_resnet.named_parameters() if not name.startswith("fc.")
]
optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-4},
        {"params": model_resnet.fc.parameters(), "lr": 1e-3},
    ],
    weight_decay=0.01,
)
epochs = 25
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

train_losses, val_losses = [], []
val_accuracies, val_balanced, val_macro_f1s = [], [], []
best_macro_f1 = -1.0
best_val_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
patience = 5
checkpoint_path = here / "best_resnet_sampler.pth"

train_resized = [Image.open(path).convert("RGB").resize((224, 224)) for path in train_dataset.df["path"]]
val_resized = [Image.open(path).convert("RGB").resize((224, 224)) for path in val_dataset.df["path"]]
train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CachedSkin(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), int(self.labels[idx])

train_labels = train_dataset.df["dx"].map(LABEL_MAP).tolist()
val_labels = val_dataset.df["dx"].map(LABEL_MAP).tolist()
train_loader = DataLoader(
    CachedSkin(train_resized, train_labels, train_aug),
    batch_size=32,
    sampler=sampler,
    shuffle=False,
)
val_loader = DataLoader(
    CachedSkin(val_resized, val_labels, eval_tensor),
    batch_size=32,
    shuffle=False,
)
print(f"Cached images: train {len(train_resized)}, val {len(val_resized)}", flush=True)

for epoch in range(1, epochs + 1):
    model_resnet.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model_resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    model_resnet.eval()
    val_loss_sum = 0.0
    pred_list, label_list = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model_resnet(images)
            val_loss_sum += criterion(outputs, labels).item()
            pred_list.extend(outputs.argmax(dim=1).cpu().tolist())
            label_list.extend(labels.cpu().tolist())
    val_loss = val_loss_sum / len(val_loader)
    val_acc = accuracy_score(label_list, pred_list)
    val_bal = balanced_accuracy_score(label_list, pred_list)
    val_f1 = f1_score(label_list, pred_list, average="macro", zero_division=0)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_balanced.append(val_bal)
    val_macro_f1s.append(val_f1)
    scheduler.step()

    print(
        f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
        f"Val Bal Acc: {val_bal:.4f} | Val Macro F1: {val_f1:.4f}",
        flush=True,
    )

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        best_val_acc = val_acc
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model_resnet.state_dict(), checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch}", flush=True)
            break

epoch_axis = range(1, len(train_losses) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epoch_axis, train_losses, label="Train loss")
axes[0].plot(epoch_axis, val_losses, label="Val loss")
axes[0].set_title("ResNet50 sampler only — Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[1].plot(epoch_axis, val_macro_f1s, label="Val macro F1", color="green")
axes[1].set_title("ResNet50 sampler only — Validation macro F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()
fig.tight_layout()
fig.savefig(here / "resnet_sampler_curves.png", dpi=150)
plt.show()

model_resnet.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
print(f"Best epoch: {best_epoch}")
print(f"Best val accuracy: {best_val_acc:.4f}")
print(f"Best val macro F1: {best_macro_f1:.4f}")


Device: cuda
Loss: unweighted cross-entropy, label smoothing 0.1
Cached images: train 6981, val 1532
Epoch 1/25 | Train Loss: 1.0213 | Val Loss: 1.0550 | Val Acc: 0.7102 | Val Bal Acc: 0.6856 | Val Macro F1: 0.6136
Epoch 2/25 | Train Loss: 0.7821 | Val Loss: 1.0845 | Val Acc: 0.6958 | Val Bal Acc: 0.6460 | Val Macro F1: 0.5955
Epoch 3/25 | Train Loss: 0.7167 | Val Loss: 1.1463 | Val Acc: 0.7174 | Val Bal Acc: 0.6593 | Val Macro F1: 0.5922
Epoch 4/25 | Train Loss: 0.6863 | Val Loss: 1.0756 | Val Acc: 0.7050 | Val Bal Acc: 0.6847 | Val Macro F1: 0.6071
Epoch 5/25 | Train Loss: 0.6271 | Val Loss: 0.9146 | Val Acc: 0.7970 | Val Bal Acc: 0.6792 | Val Macro F1: 0.6784
Epoch 6/25 | Train Loss: 0.5955 | Val Loss: 0.9001 | Val Acc: 0.8061 | Val Bal Acc: 0.6453 | Val Macro F1: 0.6561
Epoch 7/25 | Train Loss: 0.5743 | Val Loss: 0.8713 | Val Acc: 0.8316 | Val Bal Acc: 0.6716 | Val Macro F1: 0.6869
Epoch 8/25 | Train Loss: 0.5565 | Val Loss: 0.9437 | Val Acc: 0.7872 | Val Bal Acc: 0.6576 | Val Macr

## Task 6 — Fresh ViT-B/16

ViT-B/16 starts from pretrained weights with `drop_path_rate=0.1`. The training sampler stays. The loss is ordinary cross-entropy with label smoothing 0.1 and no class weights. The checkpoint is `best_vit_fair.pth`, chosen by validation macro F1. The test set is not used.

Epochs 1–5 ran at batch size 32. Epoch 6 hit a CUDA error, so training resumed from that epoch-5 checkpoint at batch size 16 with the same learning rate. The run then went through epoch 25.

In [ ]:
import math
import matplotlib.pyplot as plt
import timm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
print("Loss: unweighted cross-entropy, label smoothing", criterion.label_smoothing)

model_vit = timm.create_model(
    "vit_base_patch16_224", pretrained=True, num_classes=7, drop_path_rate=0.1
)
model_vit = model_vit.to(device)

optimizer = torch.optim.AdamW(model_vit.parameters(), lr=2e-5, weight_decay=0.05)
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1 / 3, end_factor=1.0, total_iters=3
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=22)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[3]
)

epochs = 25
patience = 5
best_macro_f1 = -1.0
best_val_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
checkpoint_path = here / "best_vit_fair.pth"
train_losses, val_losses = [], []
val_accuracies, val_balanced, val_macro_f1s = [], [], []

train_resized = [Image.open(path).convert("RGB").resize((224, 224)) for path in train_dataset.df["path"]]
val_resized = [Image.open(path).convert("RGB").resize((224, 224)) for path in val_dataset.df["path"]]
train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CachedSkin(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), int(self.labels[idx])

train_labels = train_dataset.df["dx"].map(LABEL_MAP).tolist()
val_labels = val_dataset.df["dx"].map(LABEL_MAP).tolist()
train_loader = DataLoader(
    CachedSkin(train_resized, train_labels, train_aug),
    batch_size=16,  # epochs 1-5 used 32; resumed at 16 after a CUDA error
    sampler=sampler,
    shuffle=False,
)
val_loader = DataLoader(
    CachedSkin(val_resized, val_labels, eval_tensor),
    batch_size=16,
    shuffle=False,
)
print(f"Cached images: train {len(train_resized)}, val {len(val_resized)}", flush=True)

stop_for_review = False
for epoch in range(1, epochs + 1):
    model_vit.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model_vit(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    model_vit.eval()
    val_loss_sum = 0.0
    pred_list, label_list = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model_vit(images)
            val_loss_sum += criterion(outputs, labels).item()
            pred_list.extend(outputs.argmax(dim=1).cpu().tolist())
            label_list.extend(labels.cpu().tolist())
    val_loss = val_loss_sum / len(val_loader)
    val_acc = accuracy_score(label_list, pred_list)
    val_bal = balanced_accuracy_score(label_list, pred_list)
    val_f1 = f1_score(label_list, pred_list, average="macro", zero_division=0)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_balanced.append(val_bal)
    val_macro_f1s.append(val_f1)

    print(
        f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
        f"Val Bal Acc: {val_bal:.4f} | Val Macro F1: {val_f1:.4f}",
        flush=True,
    )

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        best_val_acc = val_acc
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model_vit.state_dict(), checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch}", flush=True)
            break

    if epoch == 5 and train_loss < 0.15 and val_f1 < 0.55:
        stop_for_review = True
        print(
            "Stopping: train loss collapsed by epoch 5 while val macro F1 stayed under 0.55. "
            "Learning rate was not changed.",
            flush=True,
        )
        break

if not stop_for_review:
    epoch_axis = range(1, len(train_losses) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epoch_axis, train_losses, label="Train loss")
    axes[0].plot(epoch_axis, val_losses, label="Val loss")
    axes[0].set_title("ViT-B/16 — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[1].plot(epoch_axis, val_macro_f1s, label="Val macro F1", color="green")
    axes[1].set_title("ViT-B/16 — Validation macro F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Macro F1")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(here / "vit_fair_curves.png", dpi=150)
    plt.show()

    model_vit.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    print(f"Best epoch: {best_epoch}")
    print(f"Best val accuracy: {best_val_acc:.4f}")
    print(f"Best val macro F1: {best_macro_f1:.4f}")


Device: cuda
Loss: unweighted cross-entropy, label smoothing 0.1
Cached images: train 6981, val 1532
Epoch 1/25 | Train Loss: 1.2690 | Val Loss: 1.1271 | Val Acc: 0.6762 | Val Bal Acc: 0.6529 | Val Macro F1: 0.5492
Epoch 2/25 | Train Loss: 0.8885 | Val Loss: 0.9138 | Val Acc: 0.7866 | Val Bal Acc: 0.6632 | Val Macro F1: 0.6239
Epoch 3/25 | Train Loss: 0.7733 | Val Loss: 0.9165 | Val Acc: 0.7552 | Val Bal Acc: 0.6723 | Val Macro F1: 0.6569
Epoch 4/25 | Train Loss: 0.7082 | Val Loss: 1.0816 | Val Acc: 0.6723 | Val Bal Acc: 0.6660 | Val Macro F1: 0.6077
Epoch 5/25 | Train Loss: 0.6417 | Val Loss: 0.8808 | Val Acc: 0.8133 | Val Bal Acc: 0.6927 | Val Macro F1: 0.6712
Epoch 6/25 | Train Loss: 0.6554 | Val Loss: 1.1167 | Val Acc: 0.6984 | Val Bal Acc: 0.6955 | Val Macro F1: 0.6028
Epoch 7/25 | Train Loss: 0.6129 | Val Loss: 0.8812 | Val Acc: 0.8035 | Val Bal Acc: 0.7164 | Val Macro F1: 0.6931
Epoch 8/25 | Train Loss: 0.5873 | Val Loss: 0.8755 | Val Acc: 0.8035 | Val Bal Acc: 0.7152 | Val Macr

## Task 7 — Official test comparison

Each checkpoint is scored once on `split_test.csv` with the eval transform. ResNet is `best_resnet_sampler.pth`. ViT is `best_vit_fair.pth`. `best_resnet_fair.pth` is not loaded. The Task 1 baseline used a different image-level split, so it is context and not a paired leaderboard.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import timm
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
)
from torchvision.models import resnet50

CLASS_NAMES = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
MEL_INDEX = LABEL_MAP["mel"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Loaded checkpoints: best_resnet_sampler.pth and best_vit_fair.pth")
print("Not loaded: best_resnet_fair.pth")

test_resized = [
    Image.open(path).convert("RGB").resize((224, 224)) for path in test_dataset.df["path"]
]
test_labels = test_dataset.df["dx"].map(LABEL_MAP).tolist()
eval_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CachedSkin(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), int(self.labels[idx])


test_eval_loader = DataLoader(
    CachedSkin(test_resized, test_labels, eval_tensor),
    batch_size=16,
    shuffle=False,
)
print(f"Test images: {len(test_resized)}. Eval transform only. One pass per checkpoint.")


def predict_once(model):
    model.eval()
    label_list, pred_list, prob_list = [], [], []
    with torch.no_grad():
        for images, labels in test_eval_loader:
            probabilities = torch.softmax(model(images.to(device)), dim=1).cpu()
            label_list.extend(labels.tolist())
            pred_list.extend(probabilities.argmax(dim=1).tolist())
            prob_list.append(probabilities.numpy())
    return np.array(label_list), np.array(pred_list), np.concatenate(prob_list)


def score_predictions(labels, predictions, probabilities):
    report = classification_report(
        labels,
        predictions,
        labels=list(range(7)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    matrix = confusion_matrix(labels, predictions, labels=list(range(7)))
    pairs = []
    for true_idx in range(7):
        for pred_idx in range(7):
            if true_idx == pred_idx:
                continue
            count = int(matrix[true_idx, pred_idx])
            if count:
                pairs.append((count, CLASS_NAMES[true_idx], CLASS_NAMES[pred_idx]))
    pairs.sort(reverse=True)
    melanoma_true = (labels == MEL_INDEX).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, predictions, average="weighted", zero_division=0),
        "report": report,
        "melanoma_recall": report["mel"]["recall"],
        "melanoma_ap": average_precision_score(melanoma_true, probabilities[:, MEL_INDEX]),
        "confusion": matrix,
        "pairs": pairs[:10],
        "labels": labels,
        "probabilities": probabilities,
    }


def print_model_report(name, metrics):
    print(f"\n{name}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Balanced accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
    print(
        classification_report(
            metrics["labels"],
            metrics["probabilities"].argmax(axis=1),
            labels=list(range(7)),
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0,
        )
    )
    print(f"Melanoma recall: {metrics['melanoma_recall']:.4f}")
    print(f"Melanoma average precision: {metrics['melanoma_ap']:.4f}")
    print("Top 10 confusion pairs (true -> predicted, count):")
    for count, true_name, pred_name in metrics["pairs"]:
        print(f"  {true_name} -> {pred_name}: {count}")


def parameter_megabytes(model):
    count = sum(parameter.numel() for parameter in model.parameters())
    return count * 4 / (1024 ** 2)


def milliseconds_per_image(model):
    image = torch.randn(1, 3, 224, 224, device=device)
    model.eval()
    with torch.no_grad():
        model(image)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(100):
            model(image)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return (elapsed / 100) * 1000


model_resnet = resnet50(weights=None)
model_resnet.fc = torch.nn.Linear(2048, 7)
model_resnet.load_state_dict(
    torch.load(here / "best_resnet_sampler.pth", map_location=device, weights_only=True)
)
model_resnet = model_resnet.to(device)
resnet_size_mb = parameter_megabytes(model_resnet)
resnet_ms = milliseconds_per_image(model_resnet)
resnet_labels, resnet_preds, resnet_probs = predict_once(model_resnet)
resnet_metrics = score_predictions(resnet_labels, resnet_preds, resnet_probs)
del model_resnet
torch.cuda.empty_cache()

model_vit = timm.create_model(
    "vit_base_patch16_224", pretrained=False, num_classes=7, drop_path_rate=0.1
)
model_vit.load_state_dict(
    torch.load(here / "best_vit_fair.pth", map_location=device, weights_only=True)
)
model_vit = model_vit.to(device)
vit_size_mb = parameter_megabytes(model_vit)
vit_ms = milliseconds_per_image(model_vit)
vit_labels, vit_preds, vit_probs = predict_once(model_vit)
vit_metrics = score_predictions(vit_labels, vit_preds, vit_probs)
del model_vit
torch.cuda.empty_cache()

print_model_report("ResNet50 best_resnet_sampler.pth", resnet_metrics)
print_model_report("ViT-B/16 best_vit_fair.pth", vit_metrics)
print(f"\nResNet50 size: {resnet_size_mb:.1f} MB")
print(f"ViT-B/16 size: {vit_size_mb:.1f} MB")
print(f"ResNet50 inference: {resnet_ms:.2f} ms/image")
print(f"ViT-B/16 inference: {vit_ms:.2f} ms/image")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, metrics, title in (
    (axes[0], resnet_metrics, "ResNet50"),
    (axes[1], vit_metrics, "ViT-B/16"),
):
    image = axis.imshow(metrics["confusion"], cmap="Blues")
    axis.set_title(title)
    axis.set_xticks(range(7), CLASS_NAMES, rotation=45)
    axis.set_yticks(range(7), CLASS_NAMES)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    for row in range(7):
        for col in range(7):
            axis.text(col, row, int(metrics["confusion"][row, col]), ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=axes)
fig.tight_layout()
fig.savefig(here / "test_confusion_matrices.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metrics, title in (
    (axes[0], resnet_metrics, "ResNet50 melanoma PR"),
    (axes[1], vit_metrics, "ViT-B/16 melanoma PR"),
):
    melanoma_true = (metrics["labels"] == MEL_INDEX).astype(int)
    precision, recall, _ = precision_recall_curve(melanoma_true, metrics["probabilities"][:, MEL_INDEX])
    axis.plot(recall, precision)
    axis.set_title(f"{title} (AP {metrics['melanoma_ap']:.4f})")
    axis.set_xlabel("Recall")
    axis.set_ylabel("Precision")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(here / "test_melanoma_pr_curves.png", dpi=150)
plt.show()

comparison = {
    "model": ["ResNet50", "ViT-B/16"],
    "checkpoint": ["best_resnet_sampler.pth", "best_vit_fair.pth"],
    "accuracy": [resnet_metrics["accuracy"], vit_metrics["accuracy"]],
    "balanced_accuracy": [resnet_metrics["balanced_accuracy"], vit_metrics["balanced_accuracy"]],
    "macro_f1": [resnet_metrics["macro_f1"], vit_metrics["macro_f1"]],
    "weighted_f1": [resnet_metrics["weighted_f1"], vit_metrics["weighted_f1"]],
    "melanoma_recall": [resnet_metrics["melanoma_recall"], vit_metrics["melanoma_recall"]],
    "melanoma_ap": [resnet_metrics["melanoma_ap"], vit_metrics["melanoma_ap"]],
    "size_mb": [resnet_size_mb, vit_size_mb],
    "ms_per_image": [resnet_ms, vit_ms],
}
comparison_table = pd.DataFrame(comparison)
print("\nTest comparison")
print(comparison_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(
    "The Task 1 baseline used a different image-level split, so it is context and not a paired leaderboard."
)


Device: cuda
Loaded checkpoints: best_resnet_sampler.pth and best_vit_fair.pth
Not loaded: best_resnet_fair.pth
Test images: 1502. Eval transform only. One pass per checkpoint.

ResNet50 best_resnet_sampler.pth
Accuracy: 0.8069
Balanced accuracy: 0.7099
Macro F1: 0.7095
Weighted F1: 0.8043
              precision    recall  f1-score   support

          nv     0.8998    0.9124    0.9060      1004
         mel     0.6000    0.4671    0.5253       167
         bkl     0.5740    0.5808    0.5774       167
         bcc     0.5895    0.7887    0.6747        71
       akiec     0.6154    0.6154    0.6154        52
        vasc     1.0000    0.9048    0.9500        21
          df     0.7368    0.7000    0.7179        20

    accuracy                         0.8069      1502
   macro avg     0.7165    0.7099    0.7095      1502
weighted avg     0.8050    0.8069    0.8043      1502

Melanoma recall: 0.4671
Melanoma average precision: 0.5836
Top 10 confusion pairs (true -> predicted, count):
  

## Task 8 — Melanoma operating point

The threshold is chosen on validation only. If P(mel) is at least the threshold, the prediction is melanoma. Otherwise the prediction is the argmax among the other six classes. The chosen threshold is applied once to the test set. The Task 7 argmax report is unchanged. `best_resnet_fair.pth` is not loaded.

In [ ]:
import numpy as np
import timm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torchvision.models import resnet50

MEL_INDEX = LABEL_MAP["mel"]
CLASS_NAMES = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
THRESHOLDS = [round(0.05 * step, 2) for step in range(1, 15)]
TASK7_TEST_MELANOMA_RECALL = {
    "ResNet50": 0.4671,
    "ViT-B/16": 0.6048,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Not loaded: best_resnet_fair.pth")

eval_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CachedSkin(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), int(self.labels[idx])


def cached_loader(frame):
    images = [Image.open(path).convert("RGB").resize((224, 224)) for path in frame["path"]]
    labels = frame["dx"].map(LABEL_MAP).tolist()
    return DataLoader(CachedSkin(images, labels, eval_tensor), batch_size=16, shuffle=False)


def predict_probabilities(model, loader):
    model.eval()
    labels, probabilities = [], []
    with torch.no_grad():
        for images, batch_labels in loader:
            probabilities.append(torch.softmax(model(images.to(device)), dim=1).cpu().numpy())
            labels.extend(batch_labels.tolist())
    return np.array(labels), np.concatenate(probabilities)


def threshold_predictions(probabilities, threshold):
    other = probabilities.copy()
    other[:, MEL_INDEX] = -1.0
    below = other.argmax(axis=1)
    return np.where(probabilities[:, MEL_INDEX] >= threshold, MEL_INDEX, below)


def operating_metrics(labels, predictions):
    melanoma = (labels == MEL_INDEX).astype(int)
    predicted_melanoma = (predictions == MEL_INDEX).astype(int)
    return {
        "melanoma_recall": recall_score(melanoma, predicted_melanoma, zero_division=0),
        "melanoma_precision": precision_score(melanoma, predicted_melanoma, zero_division=0),
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
    }


def choose_threshold(rows):
    eligible = [row for row in rows if row["melanoma_precision"] >= 0.40]
    if eligible:
        eligible.sort(key=lambda row: (row["melanoma_recall"], row["melanoma_precision"], row["threshold"]), reverse=True)
        return eligible[0], "highest validation melanoma recall with precision at least 0.40"
    rows_by_f1 = sorted(rows, key=lambda row: (row["melanoma_f1"], row["threshold"]), reverse=True)
    return rows_by_f1[0], "no threshold reached precision 0.40; best validation melanoma F1"


val_loader = cached_loader(val_dataset.df)
test_loader_eval = cached_loader(test_dataset.df)
print(f"Validation images: {len(val_dataset)}. Test images: {len(test_dataset)}.")

checkpoints = [
    (
        "ResNet50",
        "best_resnet_sampler.pth",
        lambda: resnet50(weights=None),
        lambda model: setattr(model, "fc", torch.nn.Linear(2048, 7)),
    ),
    (
        "ViT-B/16",
        "best_vit_fair.pth",
        lambda: timm.create_model(
            "vit_base_patch16_224", pretrained=False, num_classes=7, drop_path_rate=0.1
        ),
        lambda model: None,
    ),
]

chosen = {}
for model_name, filename, build, adjust in checkpoints:
    model = build()
    adjust(model)
    model.load_state_dict(torch.load(here / filename, map_location=device, weights_only=True))
    model = model.to(device)
    val_labels, val_probabilities = predict_probabilities(model, val_loader)
    test_labels, test_probabilities = predict_probabilities(model, test_loader_eval)
    del model
    torch.cuda.empty_cache()

    print(f"\n{model_name} validation sweep")
    rows = []
    for threshold in THRESHOLDS:
        metrics = operating_metrics(val_labels, threshold_predictions(val_probabilities, threshold))
        melanoma_f1_denom = metrics["melanoma_precision"] + metrics["melanoma_recall"]
        melanoma_f1 = 0.0 if melanoma_f1_denom == 0 else 2 * metrics["melanoma_precision"] * metrics["melanoma_recall"] / melanoma_f1_denom
        row = {"threshold": threshold, "melanoma_f1": melanoma_f1, **metrics}
        rows.append(row)
        print(
            f"threshold {threshold:.2f} | melanoma recall {metrics['melanoma_recall']:.4f} | "
            f"melanoma precision {metrics['melanoma_precision']:.4f} | "
            f"accuracy {metrics['accuracy']:.4f} | macro F1 {metrics['macro_f1']:.4f}"
        )
    selected, reason = choose_threshold(rows)
    print(f"Chosen threshold {selected['threshold']:.2f}: {reason}")

    argmax_metrics = operating_metrics(test_labels, test_probabilities.argmax(axis=1))
    threshold_metrics = operating_metrics(
        test_labels, threshold_predictions(test_probabilities, selected["threshold"])
    )
    stored_recall = TASK7_TEST_MELANOMA_RECALL[model_name]
    if abs(argmax_metrics["melanoma_recall"] - stored_recall) > 0.00015:
        raise RuntimeError(
            f"{model_name} argmax melanoma recall {argmax_metrics['melanoma_recall']:.4f} "
            f"does not match Task 7 {stored_recall:.4f}"
        )
    chosen[model_name] = {
        "threshold": selected["threshold"],
        "reason": reason,
        "argmax": argmax_metrics,
        "thresholded": threshold_metrics,
    }

print("\nTest operating points")
for model_name, result in chosen.items():
    print(f"\n{model_name} threshold {result['threshold']:.2f} ({result['reason']})")
    print("rule | melanoma recall | melanoma precision | accuracy | macro F1")
    for rule in ("argmax", "thresholded"):
        metrics = result[rule]
        print(
            f"{rule} | {metrics['melanoma_recall']:.4f} | {metrics['melanoma_precision']:.4f} | "
            f"{metrics['accuracy']:.4f} | {metrics['macro_f1']:.4f}"
        )


Device: cuda
Not loaded: best_resnet_fair.pth
Validation images: 1532. Test images: 1502.

ResNet50 validation sweep
threshold 0.05 | melanoma recall 0.8092 | melanoma precision 0.4035 | accuracy 0.7670 | macro F1 0.6340
threshold 0.10 | melanoma recall 0.7341 | melanoma precision 0.4811 | accuracy 0.7996 | macro F1 0.6782
threshold 0.15 | melanoma recall 0.7052 | melanoma precision 0.5281 | accuracy 0.8094 | macro F1 0.6882
threshold 0.20 | melanoma recall 0.6763 | melanoma precision 0.5571 | accuracy 0.8153 | macro F1 0.7004
threshold 0.25 | melanoma recall 0.6647 | melanoma precision 0.5959 | accuracy 0.8225 | macro F1 0.7062
threshold 0.30 | melanoma recall 0.6416 | melanoma precision 0.6201 | accuracy 0.8238 | macro F1 0.7044
threshold 0.35 | melanoma recall 0.6127 | melanoma precision 0.6310 | accuracy 0.8231 | macro F1 0.7025
threshold 0.40 | melanoma recall 0.5954 | melanoma precision 0.6478 | accuracy 0.8244 | macro F1 0.7026
threshold 0.45 | melanoma recall 0.5896 | melanoma 

## Task 9 — Melanoma predicted as nevus

These maps explain the Task 7 argmax misses. ResNet uses Grad-CAM on `layer4[-1]` for the predicted nevus class. ViT uses the last block's attention matrix from the class token to the 196 patches. `best_resnet_fair.pth` is not loaded, and the Task 7 and Task 8 tables are unchanged.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch.nn.functional as F
from torchvision.models import resnet50

MEL_INDEX = LABEL_MAP["mel"]
NV_INDEX = LABEL_MAP["nv"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Not loaded: best_resnet_fair.pth")

eval_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CachedSkin(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), int(self.labels[idx])


test_images = [
    Image.open(path).convert("RGB").resize((224, 224)) for path in test_dataset.df["path"]
]
test_ids = test_dataset.df["image_id"].tolist()
test_labels = test_dataset.df["dx"].map(LABEL_MAP).tolist()
test_eval_loader = DataLoader(
    CachedSkin(test_images, test_labels, eval_tensor), batch_size=16, shuffle=False
)


def predict_probabilities(model):
    model.eval()
    probabilities = []
    with torch.no_grad():
        for images, _ in test_eval_loader:
            probabilities.append(torch.softmax(model(images.to(device)), dim=1).cpu().numpy())
    return np.concatenate(probabilities)


def miss_ids(probabilities):
    predictions = probabilities.argmax(axis=1)
    labels = np.array(test_labels)
    missed = (labels == MEL_INDEX) & (predictions == NV_INDEX)
    return set(np.array(test_ids)[missed])


def grad_cam(model, image_tensor, target_class):
    layer = model.layer4[-1]
    captured = {}

    def save_activation(_module, _inputs, output):
        captured["activation"] = output

    def save_gradient(_module, _grad_input, grad_output):
        captured["gradient"] = grad_output[0]

    forward_hook = layer.register_forward_hook(save_activation)
    backward_hook = layer.register_full_backward_hook(save_gradient)
    model.zero_grad(set_to_none=True)
    logits = model(image_tensor)
    logits[0, target_class].backward()
    activation = captured["activation"].detach()
    gradient = captured["gradient"].detach()
    weights = gradient.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * activation).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    forward_hook.remove()
    backward_hook.remove()
    return cam.cpu().numpy()


def vit_attention_map(model, image_tensor):
    attention_module = model.blocks[-1].attn
    captured = {}
    original_forward = attention_module.forward

    def forward_with_weights(x, attn_mask=None, is_causal=False):
        batch, tokens, _channels = x.shape
        gate = attention_module.gate(x).sigmoid() if attention_module.gate is not None else None
        qkv = attention_module.qkv(x).reshape(
            batch, tokens, 3, attention_module.num_heads, attention_module.head_dim
        ).permute(2, 0, 3, 1, 4)
        query, key, value = qkv.unbind(0)
        query, key = attention_module.q_norm(query), attention_module.k_norm(key)
        query = query * attention_module.scale
        weights = query @ key.transpose(-2, -1)
        weights = weights.softmax(dim=-1)
        captured["weights"] = weights.detach()
        hidden = attention_module.attn_drop(weights) @ value
        hidden = hidden.transpose(1, 2).reshape(batch, tokens, attention_module.attn_dim)
        hidden = attention_module.norm(hidden)
        if gate is not None:
            hidden = hidden * gate
        return attention_module.proj_drop(attention_module.proj(hidden))

    attention_module.forward = forward_with_weights
    model.eval()
    with torch.no_grad():
        logits = model(image_tensor)
    attention_module.forward = original_forward
    weights = captured["weights"]
    if weights.shape[-2:] != (197, 197):
        raise RuntimeError(f"Expected attention matrix 197x197, got {tuple(weights.shape)}")
    patches = weights[0, :, 0, 1:].mean(dim=0).reshape(14, 14)
    patches = (patches - patches.min()) / (patches.max() - patches.min() + 1e-8)
    resized = F.interpolate(patches[None, None], size=(224, 224), mode="bilinear", align_corners=False)
    return resized[0, 0].cpu().numpy(), tuple(weights.shape), logits


def center_fraction(heat):
    height, width = heat.shape
    top, bottom = height // 4, 3 * height // 4
    left, right = width // 4, 3 * width // 4
    return float(heat[top:bottom, left:right].sum() / (heat.sum() + 1e-8))


def overlay(image, heat):
    color = plt.cm.jet(heat)[:, :, :3]
    base = np.asarray(image).astype(np.float32) / 255.0
    return np.clip(0.55 * base + 0.45 * color, 0, 1)


model_resnet = resnet50(weights=None)
model_resnet.fc = torch.nn.Linear(2048, 7)
model_resnet.load_state_dict(
    torch.load(here / "best_resnet_sampler.pth", map_location=device, weights_only=True)
)
model_resnet = model_resnet.to(device)
resnet_probabilities = predict_probabilities(model_resnet)
resnet_misses = miss_ids(resnet_probabilities)

model_vit = timm.create_model(
    "vit_base_patch16_224", pretrained=False, num_classes=7, drop_path_rate=0.1
)
model_vit.load_state_dict(torch.load(here / "best_vit_fair.pth", map_location=device, weights_only=True))
model_vit = model_vit.to(device)
vit_probabilities = predict_probabilities(model_vit)
vit_misses = miss_ids(vit_probabilities)

print(f"ResNet mel -> nv: {len(resnet_misses)}")
print(f"ViT mel -> nv: {len(vit_misses)}")
if len(resnet_misses) != 61 or len(vit_misses) != 54:
    raise RuntimeError("Miss counts do not match the Task 7 confusion pairs")
shared = sorted(resnet_misses & vit_misses)
print(f"Missed by both models: {len(shared)}")
selected_ids = shared[:8]
print("Shown image ids:", ", ".join(selected_ids))

id_to_index = {image_id: index for index, image_id in enumerate(test_ids)}
fig, axes = plt.subplots(8, 3, figsize=(9, 24))
attention_shape = None
for row, image_id in enumerate(selected_ids):
    index = id_to_index[image_id]
    image = test_images[index]
    tensor = eval_tensor(image).unsqueeze(0).to(device)
    model_resnet.eval()
    cam = grad_cam(model_resnet, tensor, NV_INDEX)
    attention, attention_shape, _logits = vit_attention_map(model_vit, tensor)
    resnet_mel = resnet_probabilities[index, MEL_INDEX]
    vit_mel = vit_probabilities[index, MEL_INDEX]
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(
        f"{image_id}\ntrue mel, pred nv\nP(mel) R {resnet_mel:.2f} / V {vit_mel:.2f}",
        fontsize=8,
    )
    axes[row, 1].imshow(overlay(image, cam))
    axes[row, 1].set_title(f"ResNet Grad-CAM\ncenter mass {center_fraction(cam):.2f}", fontsize=8)
    axes[row, 2].imshow(overlay(image, attention))
    axes[row, 2].set_title(f"ViT attention\ncenter mass {center_fraction(attention):.2f}", fontsize=8)
    for column in range(3):
        axes[row, column].axis("off")
    print(
        f"{image_id} | true mel | pred nv | ResNet P(mel) {resnet_mel:.4f} | "
        f"ViT P(mel) {vit_mel:.4f} | Grad-CAM center {center_fraction(cam):.2f} | "
        f"attention center {center_fraction(attention):.2f}"
    )

print(f"ViT attention matrix shape: {attention_shape}")
fig.tight_layout()
fig.savefig(here / "task9_mel_to_nv.png", dpi=120)
plt.show()
del model_resnet, model_vit
torch.cuda.empty_cache()


Device: cuda
Not loaded: best_resnet_fair.pth
ResNet mel -> nv: 61
ViT mel -> nv: 54
Missed by both models: 42
Shown image ids: ISIC_0024367, ISIC_0024586, ISIC_0024700, ISIC_0025316, ISIC_0025450, ISIC_0025783, ISIC_0026369, ISIC_0026802
ISIC_0024367 | true mel | pred nv | ResNet P(mel) 0.0581 | ViT P(mel) 0.0235 | Grad-CAM center 0.77 | attention center 0.58
ISIC_0024586 | true mel | pred nv | ResNet P(mel) 0.0120 | ViT P(mel) 0.0121 | Grad-CAM center 0.69 | attention center 0.65
ISIC_0024700 | true mel | pred nv | ResNet P(mel) 0.1661 | ViT P(mel) 0.1970 | Grad-CAM center 0.46 | attention center 0.35
ISIC_0025316 | true mel | pred nv | ResNet P(mel) 0.0557 | ViT P(mel) 0.0214 | Grad-CAM center 0.66 | attention center 0.49
ISIC_0025450 | true mel | pred nv | ResNet P(mel) 0.0039 | ViT P(mel) 0.0126 | Grad-CAM center 0.64 | attention center 0.46
ISIC_0025783 | true mel | pred nv | ResNet P(mel) 0.0286 | ViT P(mel) 0.0129 | Grad-CAM center 0.55 | attention center 0.37
ISIC_0026369 | tr

On these eight shared misses, ResNet Grad-CAM for the predicted nevus class sits on the pigmented lesion or on a ring at its border. The smaller round lesions light up in the dark center; the larger irregular ones spread toward the edge of the spot. ViT attention is a different pattern: the class token weights a few patches on the lesion and many scattered patches on the surrounding skin and the corners. Hair and ruler marks are not the main highlighted region in this set.

## Task 10 — Preprocessing ablation

Train and validation images are blurred, contrast-enhanced, and hair-inpainted in memory before the usual augmentation. The official checkpoints stay unchanged. The test set runs only if an ablation beats the plain-image validation macro F1 of the same architecture.


In [ ]:
import cv2
import numpy as np
import timm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from torch.utils.data import WeightedRandomSampler
from torchvision.models import ResNet50_Weights, resnet50

MEL_INDEX = LABEL_MAP["mel"]
CLASS_ORDER = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
PLAIN_MACRO_F1 = {"ResNet50": 0.7075, "ViT-B/16": 0.7339}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, flush=True)
print("Not loaded: best_resnet_fair.pth", flush=True)
print("Official checkpoints will not be overwritten", flush=True)

eval_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
hair_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))


class CachedSkin(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), int(self.labels[idx])


def preprocess_image(image):
    img = np.asarray(image.convert("RGB").resize((224, 224)))
    img = cv2.GaussianBlur(img, (5, 5), 0)
    light, green, blue = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2LAB))
    light = clahe.apply(light)
    img = cv2.cvtColor(cv2.merge((light, green, blue)), cv2.COLOR_LAB2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, hair_kernel)
    _, hair_mask = cv2.threshold(blackhat, 25, 255, cv2.THRESH_BINARY)
    img = cv2.inpaint(img, hair_mask, 3, cv2.INPAINT_TELEA)
    return Image.fromarray(img)


def cache_plain(frame):
    return [Image.open(path).convert("RGB").resize((224, 224)) for path in frame["path"]]


def cache_preprocessed(frame, name):
    images = []
    for index, path in enumerate(frame["path"], start=1):
        images.append(preprocess_image(Image.open(path)))
        if index % 1000 == 0 or index == len(frame):
            print(f"Preprocessed {name} {index}/{len(frame)}", flush=True)
    return images


def make_loader(images, labels, transform, batch_size, sampler=None):
    return DataLoader(
        CachedSkin(images, labels, transform),
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
    )


def melanoma_recall(labels, predictions):
    true_melanoma = (np.asarray(labels) == MEL_INDEX).astype(int)
    predicted_melanoma = (np.asarray(predictions) == MEL_INDEX).astype(int)
    return recall_score(true_melanoma, predicted_melanoma, zero_division=0)


def predict(model, loader):
    model.eval()
    labels, predictions = [], []
    loss_sum = 0.0
    criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
    with torch.no_grad():
        for images, batch_labels in loader:
            images = images.to(device)
            batch_labels = batch_labels.to(device)
            logits = model(images)
            loss_sum += criterion(logits, batch_labels).item()
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
            labels.extend(batch_labels.cpu().tolist())
    return labels, predictions, loss_sum / len(loader)


def score(labels, predictions, loss):
    return {
        "loss": loss,
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "melanoma_recall": melanoma_recall(labels, predictions),
    }


def load_resnet(path):
    model = resnet50(weights=None)
    model.fc = torch.nn.Linear(2048, 7)
    model.load_state_dict(torch.load(here / path, map_location=device, weights_only=True))
    return model.to(device)


def load_vit(path, pretrained):
    model = timm.create_model(
        "vit_base_patch16_224", pretrained=pretrained, num_classes=7, drop_path_rate=0.1
    )
    if path is not None:
        model.load_state_dict(torch.load(here / path, map_location=device, weights_only=True))
    return model.to(device)


val_labels = val_dataset.df["dx"].map(LABEL_MAP).tolist()
plain_val_loader = make_loader(cache_plain(val_dataset.df), val_labels, eval_tensor, 16)
print("Plain-image validation melanoma recall", flush=True)
plain_melanoma_recall = {}
for model_name, filename, builder in (
    ("ResNet50", "best_resnet_sampler.pth", lambda: load_resnet("best_resnet_sampler.pth")),
    ("ViT-B/16", "best_vit_fair.pth", lambda: load_vit("best_vit_fair.pth", pretrained=False)),
):
    model = builder()
    labels, predictions, loss = predict(model, plain_val_loader)
    metrics = score(labels, predictions, loss)
    plain_melanoma_recall[model_name] = metrics["melanoma_recall"]
    print(
        f"{model_name} {filename} | val macro F1 {PLAIN_MACRO_F1[model_name]:.4f} | "
        f"val melanoma recall {metrics['melanoma_recall']:.4f}",
        flush=True,
    )
    del model
    torch.cuda.empty_cache()
del plain_val_loader

train_labels = train_dataset.df["dx"].map(LABEL_MAP).tolist()
train_counts = train_dataset.df["dx"].value_counts()
sample_weights = train_dataset.df["dx"].map(
    {name: len(train_dataset) / (7 * int(train_counts[name])) for name in CLASS_ORDER}
).to_numpy()
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
print("Preprocessing train and validation images in memory", flush=True)
train_images = cache_preprocessed(train_dataset.df, "train")
val_images = cache_preprocessed(val_dataset.df, "val")
val_loader = make_loader(val_images, val_labels, eval_tensor, 16)


def train_model(model_name, model, optimizer, scheduler, train_batch, checkpoint_name):
    criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
    train_loader = make_loader(train_images, train_labels, train_aug, train_batch, sampler)
    best_macro_f1 = -1.0
    best_metrics = None
    best_epoch = 0
    stall = 0
    checkpoint_path = here / checkpoint_name
    for epoch in range(1, 16):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)
        labels, predictions, val_loss = predict(model, val_loader)
        metrics = score(labels, predictions, val_loss)
        scheduler.step()
        print(
            f"{model_name} epoch {epoch}/15 | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {metrics['loss']:.4f} | Val Acc: {metrics['accuracy']:.4f} | "
            f"Val Bal Acc: {metrics['balanced_accuracy']:.4f} | "
            f"Val Macro F1: {metrics['macro_f1']:.4f} | "
            f"Val Melanoma Recall: {metrics['melanoma_recall']:.4f}",
            flush=True,
        )
        if metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = metrics["macro_f1"]
            best_metrics = metrics
            best_epoch = epoch
            stall = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            stall += 1
            if stall >= 5:
                print(f"{model_name} early stopping at epoch {epoch}", flush=True)
                break
    print(
        f"{model_name} best epoch {best_epoch} | val accuracy {best_metrics['accuracy']:.4f} | "
        f"val macro F1 {best_metrics['macro_f1']:.4f} | "
        f"val melanoma recall {best_metrics['melanoma_recall']:.4f}",
        flush=True,
    )
    return best_metrics


model_resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model_resnet.fc = torch.nn.Linear(2048, 7)
model_resnet = model_resnet.to(device)
resnet_optimizer = torch.optim.AdamW(
    [
        {"params": [p for n, p in model_resnet.named_parameters() if not n.startswith("fc.")], "lr": 1e-4},
        {"params": model_resnet.fc.parameters(), "lr": 1e-3},
    ],
    weight_decay=0.01,
)
resnet_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(resnet_optimizer, T_max=15)
resnet_best = train_model("ResNet50", model_resnet, resnet_optimizer, resnet_scheduler, 32, "best_resnet_preproc.pth")
del model_resnet
torch.cuda.empty_cache()

model_vit = load_vit(None, pretrained=True)
vit_optimizer = torch.optim.AdamW(model_vit.parameters(), lr=2e-5, weight_decay=0.05)
warmup = torch.optim.lr_scheduler.LinearLR(vit_optimizer, start_factor=1 / 3, end_factor=1.0, total_iters=3)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(vit_optimizer, T_max=12)
vit_scheduler = torch.optim.lr_scheduler.SequentialLR(
    vit_optimizer, schedulers=[warmup, cosine], milestones=[3]
)
vit_best = train_model("ViT-B/16", model_vit, vit_optimizer, vit_scheduler, 16, "best_vit_preproc.pth")
del model_vit
torch.cuda.empty_cache()

print("\nValidation comparison", flush=True)
print(
    "ResNet50 plain macro F1 0.7075 | plain melanoma recall "
    f"{plain_melanoma_recall['ResNet50']:.4f} | preproc macro F1 {resnet_best['macro_f1']:.4f} | "
    f"preproc melanoma recall {resnet_best['melanoma_recall']:.4f}",
    flush=True,
)
print(
    "ViT-B/16 plain macro F1 0.7339 | plain melanoma recall "
    f"{plain_melanoma_recall['ViT-B/16']:.4f} | preproc macro F1 {vit_best['macro_f1']:.4f} | "
    f"preproc melanoma recall {vit_best['melanoma_recall']:.4f}",
    flush=True,
)

if resnet_best["macro_f1"] > PLAIN_MACRO_F1["ResNet50"] or vit_best["macro_f1"] > PLAIN_MACRO_F1["ViT-B/16"]:
    print("Secondary ablation test. This does not replace the Task 7 table.", flush=True)
    test_images = cache_preprocessed(test_dataset.df, "test")
    test_labels = test_dataset.df["dx"].map(LABEL_MAP).tolist()
    test_loader_eval = make_loader(test_images, test_labels, eval_tensor, 16)
    for model_name, best, filename, builder, plain_f1 in (
        ("ResNet50", resnet_best, "best_resnet_preproc.pth", lambda: load_resnet("best_resnet_preproc.pth"), 0.7075),
        ("ViT-B/16", vit_best, "best_vit_preproc.pth", lambda: load_vit("best_vit_preproc.pth", pretrained=False), 0.7339),
    ):
        if best["macro_f1"] <= plain_f1:
            print(f"{model_name} validation macro F1 did not beat plain images. Test set not run.", flush=True)
            continue
        model = builder()
        labels, predictions, loss = predict(model, test_loader_eval)
        metrics = score(labels, predictions, loss)
        del model
        torch.cuda.empty_cache()
        print(
            f"Secondary ablation {model_name} {filename} | test accuracy {metrics['accuracy']:.4f} | "
            f"test macro F1 {metrics['macro_f1']:.4f} | test melanoma recall {metrics['melanoma_recall']:.4f}",
            flush=True,
        )
else:
    print("Neither ablation beat its plain-image validation macro F1. Test set not run.", flush=True)


## Task 11 — Results

Under a lesion-level split, how close is a properly trained ViT-B/16 to ResNet50 on HAM10000, and which errors remain?

### Split

HAM10000 has 10,015 images and 7,470 lesions. Each lesion has one diagnosis. Images were assigned by `lesion_id`, stratified with `random_state=42`, 70% train and the remaining 30% split evenly into validation and test.

| Split | Lesions | Images |
|---|---:|---:|
| Train | 5,229 | 6,981 |
| Validation | 1,120 | 1,532 |
| Test | 1,121 | 1,502 |

Lesion overlap is 0 for train–val, train–test, and val–test. All 10,015 paths were found. Missing paths: 0.

### Training

Both official models started from ImageNet weights and saw the original JPEGs, resized to 224. Training used horizontal flip, rotation of 20 degrees, and color jitter, then ImageNet normalization. The training loader used a `WeightedRandomSampler`. Validation and test loaders were unweighted and unshuffled. The loss was cross-entropy with label smoothing 0.1 and no class weights. The saved epoch is the one with the highest validation macro F1: ResNet `best_resnet_sampler.pth` at epoch 11, ViT `best_vit_fair.pth` at epoch 24.

### Official test comparison

Each checkpoint was scored once on the 1,502-image test set. These numbers are copied from the Task 7 output.

| Model | Checkpoint | Accuracy | Balanced accuracy | Macro F1 | Weighted F1 | Melanoma recall | Melanoma AP | Size (MB) | ms/image |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|
| ResNet50 | best_resnet_sampler.pth | 0.8069 | 0.7099 | 0.7095 | 0.8043 | 0.4671 | 0.5836 | 89.7 | 31.85 |
| ViT-B/16 | best_vit_fair.pth | 0.8555 | 0.7452 | 0.7632 | 0.8505 | 0.6048 | 0.6348 | 327.3 | 24.92 |

ViT is ahead on accuracy, balanced accuracy, macro F1, weighted F1, melanoma recall, and melanoma average precision. The largest miss for both models is melanoma predicted as nevus: 61 images for ResNet and 54 for ViT. Forty-two of those images were missed by both.

The Task 1 baseline used a different image-level split, so it is context and not a paired leaderboard. That split was 7,010 / 1,502 / 1,503 images, and the same lesion can appear in more than one split.

| Run | Model | Test accuracy | Macro F1 | Weighted F1 | Melanoma recall |
|---|---|---:|---:|---:|---:|
| Cross-entropy, 10 epochs | ResNet50 | 82% | 0.67 | 0.81 | 0.50 |
| Cross-entropy, 10 epochs | ViT-B/16 | 76% | 0.53 | 0.73 | 0.43 |
| Focal loss, continued | ResNet50 | 84% | 0.73 | 0.84 | 0.53 |
| Focal loss, continued | ViT-B/16 | 76% | 0.54 | 0.75 | 0.42 |
| Focal loss + MixUp, fresh | ResNet50 | 84% | 0.71 | 0.83 | 0.51 |

### Melanoma threshold

On validation, both models chose threshold 0.05: the highest melanoma recall whose melanoma precision was still at least 0.40. If the melanoma probability is at least 0.05, the prediction is melanoma. Otherwise the prediction is the argmax among the other six classes. The test comparison below is copied from the Task 8 output. The argmax row matches Task 7.

| Model | Rule | Melanoma recall | Melanoma precision | Accuracy | Macro F1 |
|---|---|---:|---:|---:|---:|
| ResNet50 | argmax | 0.4671 | 0.6000 | 0.8069 | 0.7095 |
| ResNet50 | threshold 0.05 | 0.7665 | 0.3754 | 0.7617 | 0.6682 |
| ViT-B/16 | argmax | 0.6048 | 0.7014 | 0.8555 | 0.7632 |
| ViT-B/16 | threshold 0.05 | 0.7485 | 0.4596 | 0.8142 | 0.7235 |

Calling more lesions melanoma raises recall and lowers precision, accuracy, and macro F1. ResNet test precision at this threshold is 0.3754, below the 0.40 validation rule. ViT test precision stays at 0.4596.

### Where the models look

On eight shared melanoma-to-nevus misses, ResNet Grad-CAM for the predicted nevus class sits on the pigmented lesion or on a ring at its border. ViT attention, taken from the last-block attention matrix of shape `(1, 12, 197, 197)`, is patchier: a few patches on the lesion and many on the surrounding skin and the corners. Hair and ruler marks are not the main highlighted region in this set. The figure is `task9_mel_to_nv.png`.

### Preprocessing ablation

Blur, CLAHE, and hair removal were applied in memory and compared on validation only. Copied from the Task 10 output:

| Model | Plain val macro F1 | Plain melanoma recall | Preprocessed val macro F1 | Preprocessed melanoma recall |
|---|---:|---:|---:|---:|
| ResNet50 | 0.7075 | 0.6127 | 0.6466 | 0.7110 |
| ViT-B/16 | 0.7339 | 0.6821 | 0.7015 | 0.5838 |

Preprocessing lowered validation macro F1 for both models, so the test set was not run. Task 7 stays the official comparison. The preprocessed ResNet best epoch was 10. The preprocessed ViT best epoch was 8, with early stopping at epoch 13.

### Against the predicted ranges

The ranges were for this lesion-level split.

| Model | Test accuracy | Predicted | Macro F1 | Predicted | Melanoma recall | Predicted |
|---|---:|---|---:|---|---:|---|
| ViT-B/16 | 0.8555 | 81–86%, inside | 0.7632 | 0.70–0.78, inside | 0.6048 | 0.60–0.75, inside |
| ResNet50 | 0.8069 | 80–85%, inside | 0.7095 | 0.72–0.80, short | 0.4671 | 0.60–0.75, short |

### Limitations

This is one public dataset and one split seed. There is no external clinical test set. The models are a research comparison, not a diagnostic system. Accuracy stays high partly because nevus is common: 4,683 of 6,981 training images, 1,018 of 1,532 validation images, and 1,004 of 1,502 test images. Macro F1 and melanoma recall are the numbers that carry the claim.
